In [ ]:
import numpy as np
import torch
from pathlib import Path
import problem_config as cfg
import pickle
import matplotlib.pyplot as plt


device = "cpu"
dtype = cfg.dtype

# --- Paths relative to problem_config.py ---
base_dir   = Path(cfg.__file__).parent.resolve()
data_dir   = base_dir / "problem_data"
result_dir = base_dir / "result"
result_dir.mkdir(exist_ok=True)

# --- Load data ---
y_np = np.load(data_dir / "obs_path.npy")          # (T+1, m)
y    = torch.tensor(y_np, dtype=dtype, device=device)

x_true_np = np.load(data_dir / "state_path.npy")          # (T+1, m)

# --- Matrices ---
A = cfg.A.to(device=device, dtype=dtype)
H = cfg.H.to(device=device, dtype=dtype)
Q = cfg.Q.to(device=device, dtype=dtype)
R = cfg.R.to(device=device, dtype=dtype)

n = cfg.DIM_STATE
m = cfg.DIM_OBS
T = y.shape[0]

# --- EnKF params ---
N_ens      = 200
inflation  = 1.00
denom = float(N_ens - 1)


n_rep = 10

In [ ]:
result_all = []
for rep_id in range(n_rep):
    print(f"EnKF rep {rep_id+1}/{n_rep}")
    torch.manual_seed(rep_id)

    # Prior ensemble
    m0 = torch.zeros(n, dtype=dtype, device=device)
    P0 = torch.eye(n, dtype=dtype, device=device)
    L0 = torch.linalg.cholesky(P0)
    X  = m0 + torch.randn(N_ens, n, dtype=dtype, device=device) @ L0.T

    LQ = torch.linalg.cholesky(Q)
    LR = torch.linalg.cholesky(R)

    ms = torch.zeros(T-1, n, dtype=dtype, device=device)
    Ps = torch.zeros(T-1, n, n, dtype=dtype, device=device)

    for k in range(T-1):
        xbar_a = X.mean(dim=0)
        Xp_a   = X - xbar_a
        P_a    = (Xp_a.T @ Xp_a) / denom

        ms[k] = xbar_a
        Ps[k] = P_a
        
        
        # ---- Forecast ----
        X = X @ A.T + torch.randn(N_ens, n, dtype=dtype, device=device) @ LQ.T

        # ---- Analysis ----
        xbar_f = X.mean(dim=0)
        Xp     = (X - xbar_f) * inflation

        Yf     = X @ H.T
        ybar_f = Yf.mean(dim=0)
        Yp     = Yf - ybar_f

        P_xy  = (Xp.T @ Yp) / denom
        P_yy  = (Yp.T @ Yp) / denom + R

        K = P_xy @ torch.linalg.inv(P_yy)

        y_tilde = y[k+1].unsqueeze(0).repeat(N_ens, 1) + torch.randn(N_ens, m, dtype=dtype, device=device) @ LR.T
        innov   = y_tilde - Yf
        X       = X + innov @ K.T

        # xbar_a = X.mean(dim=0)
        # Xp_a   = X - xbar_a
        # P_a    = (Xp_a.T @ Xp_a) / denom

        # ms[k] = xbar_a
        # Ps[k] = P_a
        
        
        # print error
    result_all.append((ms.cpu().numpy(), Ps.cpu().numpy()))
        
    # Save results
out_path = result_dir / "enkf.pkl"
with open(out_path, "wb") as f:
    pickle.dump(result_all, f)
print(f"Saved EnKF posterior to: {out_path}")

In [ ]:
def kl_gaussian(m0, P0, m1, P1, jitter=1e-9):
    """
    KL( N(m0,P0) || N(m1,P1) ) per step.
    Uses stable solves and adds jitter to keep PD.
    """
    n = P0.shape[-1]
    # jitter for stability
    P0j = P0.copy()
    P1j = P1.copy()
    for M in (P0j, P1j):
        M[...] = 0.5 * (M + M.transpose(0,1))  # symmetrize (if batched later)
    P0j = P0j + jitter * np.eye(n)
    P1j = P1j + jitter * np.eye(n)

    # Cholesky of P1 to get logdet and solve P1^{-1} * X
    L1 = np.linalg.cholesky(P1j)
    logdet_P1 = 2.0 * np.sum(np.log(np.diag(L1)))
    # logdet of P0
    L0 = np.linalg.cholesky(P0j)
    logdet_P0 = 2.0 * np.sum(np.log(np.diag(L0)))

    # trace(P1^{-1} P0) via solves
    # solve L1 * Y = P0  => Y, then solve L1^T * Z = Y => Z = P1^{-1} P0
    Y = np.linalg.solve(L1, P0j)
    Z = np.linalg.solve(L1.T, Y)
    tr_term = np.trace(Z)

    d = (m1 - m0).reshape(-1, 1)  # (n,1)
    # quadratic form d^T P1^{-1} d via solves
    y = np.linalg.solve(L1, d)
    q = y.T @ y

    return float(np.asarray(0.5 * (logdet_P1 - logdet_P0 - n + tr_term + q)).reshape(()))



base_dir = Path(".").resolve()
result_dir = base_dir / "result"
data_dir = base_dir / "problem_data"

# load results
kf = np.load(result_dir / "kf.npz") # ground truth
ms_kf = kf["posterior_mean"]         # (T, n)
Ps_kf = kf["posterior_cov"]          # (T, n, n)
print(f"KF:   {ms_kf.shape}, {Ps_kf.shape}")


# bench mark
with open("result/enkf.pkl", "rb") as f:
    loaded_results = pickle.load(f)


T_total, n = ms_kf.shape
t = np.arange(T_total)


In [ ]:

# 
kl_rep_all = []

for rep_id, (ms, Ps) in enumerate(loaded_results):
    # compute KL per step
    kl_all = np.zeros(T_total)
    for k in range(T_total):
        m0, P0 = ms_kf[k], Ps_kf[k]
        m, P = ms[k], Ps[k]
        kl_all[k] = kl_gaussian(m0, P0, m, P)
    kl_rep_all.append(kl_all)
kl_rep_all = np.array(kl_rep_all)  # (n_rep, T_total)
kl_mean = kl_rep_all.mean(axis=0)  # (T_total,)
kl_std  = kl_rep_all.std(axis=0)   # (T_total,)
#

In [ ]:


# plot all KL curves
plt.figure(figsize=(6,4))
for rep_id in range(kl_rep_all.shape[0]):
# for rep_id in range(1):
    plt.plot(t, kl_rep_all[rep_id], color="C0", alpha=0.3)
plt.xlabel("time step")
plt.ylabel("KL divergence")
plt.ylim(0,0.5)
plt.tight_layout()
plt.show()


qs = [0.1, 0.5, 0.90]
quantiles = np.quantile(kl_rep_all, qs, axis=0)   # shape (len(qs), 101)
q_lo, q_mid, q_hi = quantiles  # unpack


# plot all KL curves
plt.figure(figsize=(6,4))
for i, quantile in enumerate(quantiles):
    plt.plot(t, quantile, alpha=0.5,label=f"{qs[i]:.2f}")
plt.xlabel("time step")
plt.ylabel("KL divergence")
plt.ylim(0,0.5)
plt.tight_layout()
plt.show()


In [ ]:
kl_rep_all[:,0]